In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q xgboost

import os
import glob
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_score, recall_score, f1_score)

PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'notebooks/results')
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)

RANDOM_STATE = 42

In [ ]:
FEATURE_COLS_BEHAV = ['mean_speed', 'max_speed', 'std_speed', 'path_length',
                       'num_stops', 'mean_angle', 'num_clicks', 'num_points']

behav_csv = f'{PROJECT_DIR}/data/processed/behavioral_features_windowed.csv'
df_behav = pd.read_csv(behav_csv)
print(df_behav.shape[0])

X_behav = df_behav[FEATURE_COLS_BEHAV]
y_behav = df_behav['label']

X_behav_train, X_behav_test, y_behav_train, y_behav_test = train_test_split(
    X_behav, y_behav, test_size=0.2, stratify=y_behav, random_state=RANDOM_STATE
)

rf_behav = joblib.load(f'{PROJECT_DIR}/models/behavioral_rf_300.pkl')
proba_behav_test = rf_behav.predict_proba(X_behav_test)[:, list(rf_behav.classes_).index('bot')]

print(len(X_behav_test))

behav_scores_by_class = {
    'bot': proba_behav_test[(y_behav_test == 'bot').values],
    'human': proba_behav_test[(y_behav_test == 'human').values],
}
print(behav_scores_by_class['bot'].mean())
print(behav_scores_by_class['human'].mean())

In [ ]:
CTU_PATH = os.path.join(PROJECT_DIR, 'data/raw/ctu13')


def load_ctu():
    attack_file = os.path.join(CTU_PATH, 'CTU13_Attack_Traffic.csv')
    normal_file = os.path.join(CTU_PATH, 'CTU13_Normal_Traffic.csv')
    df_attack = pd.read_csv(attack_file, low_memory=False)
    df_normal = pd.read_csv(normal_file, low_memory=False)

    raw_label_variants = ['Label', 'LABEL', 'Label_original']
    df_attack = df_attack.drop(columns=[c for c in raw_label_variants if c in df_attack.columns], errors='ignore')
    df_normal = df_normal.drop(columns=[c for c in raw_label_variants if c in df_normal.columns], errors='ignore')

    common_cols = df_attack.columns.intersection(df_normal.columns)
    df_attack, df_normal = df_attack[common_cols].copy(), df_normal[common_cols].copy()
    df_attack['label'] = 1
    df_normal['label'] = 0
    df = pd.concat([df_attack, df_normal], ignore_index=True)
    return df


df_ctu = load_ctu()
df_ctu = df_ctu.replace([np.inf, -np.inf], np.nan).dropna()
y_net = df_ctu['label']
X_net = df_ctu.drop(columns=['label']).select_dtypes(include=[np.number])

print(X_net.shape[0], X_net.shape[1])

X_net_train, X_net_test, y_net_train, y_net_test = train_test_split(
    X_net, y_net, test_size=0.2, stratify=y_net, random_state=RANDOM_STATE
)

scaler_net = StandardScaler()
X_net_train_s = scaler_net.fit_transform(X_net_train)
X_net_test_s = scaler_net.transform(X_net_test)

rf_net = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                 class_weight='balanced', n_jobs=-1)
rf_net.fit(X_net_train_s, y_net_train)
proba_net_test = rf_net.predict_proba(X_net_test_s)[:, 1]

print(len(X_net_test))
print(classification_report(y_net_test, rf_net.predict(X_net_test_s), zero_division=0))

net_scores_by_class = {
    'bot': proba_net_test[(y_net_test == 1).values],
    'human': proba_net_test[(y_net_test == 0).values],
}

In [ ]:
N_SIM_PER_CLASS = 5000
rng = np.random.default_rng(RANDOM_STATE)

sim_rows = []
for cls, true_label in [('bot', 1), ('human', 0)]:
    behav_pool = behav_scores_by_class[cls]
    net_pool = net_scores_by_class[cls]
    sampled_behav = rng.choice(behav_pool, size=N_SIM_PER_CLASS, replace=True)
    sampled_net = rng.choice(net_pool, size=N_SIM_PER_CLASS, replace=True)
    for sb, sn in zip(sampled_behav, sampled_net):
        sim_rows.append({'score_behavioral': sb, 'score_network': sn, 'is_bot': true_label})

df_sim = pd.DataFrame(sim_rows)
print(len(df_sim))
print(df_sim.groupby('is_bot')[['score_behavioral', 'score_network']].mean())

In [ ]:
X_sim = df_sim[['score_behavioral', 'score_network']]
y_sim = df_sim['is_bot']

X_sim_train, X_sim_test, y_sim_train, y_sim_test = train_test_split(
    X_sim, y_sim, test_size=0.3, stratify=y_sim, random_state=RANDOM_STATE
)

results_fusion = []


def eval_strategy(name, y_true, y_pred_binary):
    acc = accuracy_score(y_true, y_pred_binary)
    prec = precision_score(y_true, y_pred_binary, zero_division=0)
    rec = recall_score(y_true, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true, y_pred_binary, zero_division=0)
    results_fusion.append({'strategy': name, 'accuracy': acc, 'precision': prec,
                            'recall': rec, 'f1': f1})
    print(name, acc, prec, rec, f1)


eval_strategy("behavioral", y_sim_test,
              (X_sim_test['score_behavioral'] >= 0.5).astype(int))

eval_strategy("network", y_sim_test,
              (X_sim_test['score_network'] >= 0.5).astype(int))

avg_score = (X_sim_test['score_behavioral'] + X_sim_test['score_network']) / 2
eval_strategy("average", y_sim_test, (avg_score >= 0.5).astype(int))

meta_model = LogisticRegression(random_state=RANDOM_STATE)
meta_model.fit(X_sim_train, y_sim_train)
y_pred_meta = meta_model.predict(X_sim_test)
eval_strategy("meta", y_sim_test, y_pred_meta)

print(meta_model.coef_[0][0], meta_model.coef_[0][1], meta_model.intercept_[0])

df_results_fusion = pd.DataFrame(results_fusion)
df_results_fusion.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'fusion_strategy_comparison.csv'), index=False)

In [ ]:
plt.figure(figsize=(10, 6))
df_plot = df_results_fusion.melt(id_vars='strategy', value_vars=['accuracy', 'precision', 'recall', 'f1'],
                                  var_name='metric', value_name='value')
ax = sns.barplot(data=df_plot, x='strategy', y='value', hue='metric')
plt.title('fusion')
plt.ylabel('value')
plt.xlabel('')
plt.ylim(0, 1.05)
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'fusion_strategy_comparison.png'), dpi=300)
plt.show()

In [ ]:
cm = confusion_matrix(y_sim_test, y_pred_meta)
print(pd.DataFrame(cm, index=['true_normal', 'true_bot'], columns=['pred_normal', 'pred_bot']))
print()
print(classification_report(y_sim_test, y_pred_meta, target_names=['normal/human', 'bot'], zero_division=0))

os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
joblib.dump(meta_model, f'{PROJECT_DIR}/models/fusion_meta_logreg.pkl')

In [ ]:
best_row = df_results_fusion.loc[df_results_fusion['f1'].idxmax()]
print(best_row['strategy'], best_row['f1'], best_row['accuracy'])